In [1]:
!nvidia-smi
!nvcc --version

Wed May 20 07:50:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
%%writefile vector_mul.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void computeDailySales(
    float* sales,
    float* prices,
    float* dailySales,
    int rows,
    int cols
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < rows * cols) {
        dailySales[idx] = sales[idx] * prices[idx];
    }
}

__global__ void computeTotalSales(float* dailySales, float* total, int size) {
    __shared__ float partialSum[256];

    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < size)
        partialSum[tid] = dailySales[idx];
    else
        partialSum[tid] = 0.0f;

    __syncthreads();

    for (int stride = blockDim.x / 2; stride > 0; stride /= 2) {
        if (tid < stride)
            partialSum[tid] += partialSum[tid + stride];
        __syncthreads();
    }

    if (tid == 0)
        atomicAdd(total, partialSum[0]);
}

const char* rowLabels[] = { "Diesel  ", "Gasoline", "Kerosene" };
const char* colLabels[] = { "Mon", "Tue", "Wed", "Thu" };

void printMatrix(const char* title, float* mat, int rows, int cols) {
    std::cout << "  " << title << "\n";
    std::cout << "            ";
    for (int j = 0; j < cols; j++)
        std::cout << colLabels[j] << "\t";
    std::cout << "\n";
    for (int i = 0; i < rows; i++) {
        std::cout << "  " << rowLabels[i] << "  ";
        for (int j = 0; j < cols; j++)
            std::cout << mat[i * cols + j] << "\t";
        std::cout << "\n";
    }
    std::cout << "\n";
}

void printRowByRow(float* sales, float* prices, float* result, int rows, int cols) {
    // "  " + rowLabel(8) + "  " + "sales:  " = 20 chars before first value
    std::cout << "                    ";
    for (int j = 0; j < cols; j++)
        std::cout << colLabels[j] << "\t\t";
    std::cout << "\n";

    for (int i = 0; i < rows; i++) {
        std::cout << "  " << rowLabels[i] << "  ";
        std::cout << "sales:  ";
        for (int j = 0; j < cols; j++)
            std::cout << sales[i * cols + j] << "\t\t";
        std::cout << "\n";

        std::cout << "            ";
        std::cout << "price:  ";
        for (int j = 0; j < cols; j++)
            std::cout << prices[i * cols + j] << "\t\t";
        std::cout << "\n";

        std::cout << "            ";
        std::cout << "result: ";
        for (int j = 0; j < cols; j++)
            std::cout << result[i * cols + j] << "\t\t";
        std::cout << "\n\n";
    }
}

int main() {
    const int rows = 3;
    const int cols = 4;
    const int size = rows * cols;

    float h_sales[size] = {
        12, 11, 4, 3,   // Diesel
        12,  8, 5, 6,   // Gasoline
         3,  7, 8, 2    // Kerosene
    };

    float h_fixedPrices[size] = {
        2, 2, 2, 2,     // Diesel    = 2
        1, 1, 1, 1,     // Gasoline  = 1
        2, 2, 2, 2      // Kerosene  = 2
    };

    float h_variablePrices[size] = {
        2, 3, 6, 7,     // Diesel
        1, 8, 3, 7,     // Gasoline
        2, 3, 5, 1      // Kerosene
    };

    float *d_sales, *d_prices, *d_dailySales, *d_total;

    cudaMalloc(&d_sales,      size * sizeof(float));
    cudaMalloc(&d_prices,     size * sizeof(float));
    cudaMalloc(&d_dailySales, size * sizeof(float));
    cudaMalloc(&d_total,      sizeof(float));

    cudaMemcpy(d_sales, h_sales, size * sizeof(float), cudaMemcpyHostToDevice);

    int blockSize = 256;
    int gridSize  = (size + blockSize - 1) / blockSize;

    // ===== PART A =====
    cudaMemcpy(d_prices, h_fixedPrices, size * sizeof(float), cudaMemcpyHostToDevice);
    computeDailySales<<<gridSize, blockSize>>>(d_sales, d_prices, d_dailySales, rows, cols);

    float h_dailySalesA[size];
    cudaMemcpy(h_dailySalesA, d_dailySales, size * sizeof(float), cudaMemcpyDeviceToHost);

    std::cout << "===== PART A: Fixed Prices =====\n\n";
    printRowByRow(h_sales, h_fixedPrices, h_dailySalesA, rows, cols);

    // ===== PART B =====
    cudaMemcpy(d_prices, h_variablePrices, size * sizeof(float), cudaMemcpyHostToDevice);
    computeDailySales<<<gridSize, blockSize>>>(d_sales, d_prices, d_dailySales, rows, cols);

    float h_dailySalesB[size];
    cudaMemcpy(h_dailySalesB, d_dailySales, size * sizeof(float), cudaMemcpyDeviceToHost);

    std::cout << "===== PART B: Varying Prices =====\n\n";
    printRowByRow(h_sales, h_variablePrices, h_dailySalesB, rows, cols);

    // ===== PART C =====
    float zero = 0.0f;
    cudaMemcpy(d_total, &zero, sizeof(float), cudaMemcpyHostToDevice);
    computeTotalSales<<<gridSize, blockSize>>>(d_dailySales, d_total, size);

    float h_total;
    cudaMemcpy(&h_total, d_total, sizeof(float), cudaMemcpyDeviceToHost);

    std::cout << "===== PART C: Total Sales =====\n\n";
    printMatrix("Daily Sales (Part B) being summed:", h_dailySalesB, rows, cols);
    std::cout << "  Total Sales = " << h_total << "\n";

    cudaFree(d_sales);
    cudaFree(d_prices);
    cudaFree(d_dailySales);
    cudaFree(d_total);

    return 0;
}

Overwriting vector_mul.cu


In [7]:
!nvcc vector_mul.cu -o vector_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [8]:
!./vector_mul

===== PART A: Fixed Prices =====

                    Mon		Tue		Wed		Thu		
  Diesel    sales:  12		11		4		3		
            price:  2		2		2		2		
            result: 24		22		8		6		

  Gasoline  sales:  12		8		5		6		
            price:  1		1		1		1		
            result: 12		8		5		6		

  Kerosene  sales:  3		7		8		2		
            price:  2		2		2		2		
            result: 6		14		16		4		

===== PART B: Varying Prices =====

                    Mon		Tue		Wed		Thu		
  Diesel    sales:  12		11		4		3		
            price:  2		3		6		7		
            result: 24		33		24		21		

  Gasoline  sales:  12		8		5		6		
            price:  1		8		3		7		
            result: 12		64		15		42		

  Kerosene  sales:  3		7		8		2		
            price:  2		3		5		1		
            result: 6		21		40		2		

===== PART C: Total Sales =====

  Daily Sales (Part B) being summed:
            Mon	Tue	Wed	Thu	
  Diesel    24	33	24	21	
  Gasoline  12	64	15	42	
  Kerosene  6	21	40	2	

  Total Sales = 304
